In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


# 1. Feature 선정 및 전처리 
def preprocess_features(df, is_train=True):
    data = df.copy()
    
    if '종목코드' not in data.columns and data.index.name == '종목코드':
        data = data.reset_index()
    elif '종목코드' not in data.columns and data.columns[0] == 'Unnamed: 0':
        data.rename(columns={data.columns[0]: '종목코드'}, inplace=True)
        
    data = data.fillna(0)
    
    my_features = ['불성실공시법인', '파산신청', '대표이사변경', '전환사채', '거래정지']
    
    # 테스트 데이터에 해당 컬럼이 없을 경우 0으로 생성
    for col in my_features:
        if col not in data.columns:
            data[col] = 0.0
            
    if is_train:
        # 훈련 시: 레이블 이상치(9, 11, 15 등) 필터링 로직 추가
        data['레이블'] = pd.to_numeric(data['레이블'], errors='coerce').fillna(0)
        
        # 정상적인 타겟(0, 1)만 남기기
        valid_data = data[data['레이블'].isin([0, 1])]
        
        X = valid_data[my_features]
        y = valid_data['레이블'].astype(int)
    else:
        # 테스트 시
        X = data[my_features]
        y = None
        if '레이블' in data.columns:
            y = pd.to_numeric(data['레이블'], errors='coerce').fillna(0).astype(int)
            
    return X, y


# 2. 모델 학습
def train_and_save_model():
    print("1. train.csv 데이터를 로드 및 정제")
    try:
        df_train = pd.read_csv('train.csv', encoding='cp949')
    except Exception:
        df_train = pd.read_csv('train.csv', encoding='utf-8')
    
    # 함수를 올바르게 호출하여 X, y를 분리
    X, y = preprocess_features(df_train, is_train=True)
    
    # 데이터 세트 분리
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print("2. 선정된 5개 Feature로 모델 학습 시작")
    model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    model.fit(X_train, y_train)
    
    # 검증 데이터 평가
    y_pred = model.predict(X_val)
    print("\n--- [검증 데이터 평가 결과 (macro avg 확인)] ---")
    print(classification_report(y_val, y_pred)) 
    
    # 모델 저장
    model_filename = 'delisting_model.pkl'
    joblib.dump(model, model_filename)
    print(f"3. 모델이 '{model_filename}' 파일에 저장됨.\n")


# 3. test_예시.xlsx 실행

def run_test_environment():
    print("4. [테스트 환경] 저장된 모델과 test.xlsx 연동 시작")
    try:
        # 1) 저장된 모델 불러오기
        loaded_model = joblib.load('delisting_model.pkl')
        
        # 2) 테스트 데이터 로드 (openpyxl 필요)
        test_df = pd.read_excel('test.xlsx')
        
        # 3) 테스트 데이터에 동일한 Feature 적용
        X_test, y_test = preprocess_features(test_df, is_train=False)
        
        # 4) 예측 수행
        predictions = loaded_model.predict(X_test)
        
        print("\ntest.xlsx 최종 평가")
        if y_test is not None:
            # 레이블이 존재하는 경우 전체 성능 지표 출력
            print(classification_report(y_test, predictions))
        else:
            # 평가용 데이터라 레이블이 숨겨져 있다면 예측값만 도출
            print("테스트 데이터에 '레이블'이 없어 예측값만 도출했습니다.")
            test_df['예측_레이블'] = predictions
            code_col = '종목코드' if '종목코드' in test_df.columns else test_df.columns[0]
            print(test_df[[code_col, '예측_레이블']].head())
            
    except FileNotFoundError:
        print("\n 'test.xlsx' 파일을 찾을 수 없습니다.")
    except Exception as e:
        print(f"\n 테스트 환경 실행 중 오류 발생: {e}")

#==========================================
if __name__ == "__main__":
    # 1. 모델 학습 및 저장
    train_and_save_model()
    
    # 2. 테스트 환경 실행
    run_test_environment()

1. train.csv 데이터를 로드 및 정제
2. 선정된 5개 Feature로 모델 학습 시작

--- [검증 데이터 평가 결과 (macro avg 확인)] ---
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       279
           1       0.77      0.91      0.83        47

    accuracy                           0.95       326
   macro avg       0.88      0.93      0.90       326
weighted avg       0.95      0.95      0.95       326

3. 모델이 'delisting_model.pkl' 파일에 저장됨.

4. [테스트 환경] 저장된 모델과 test_예시.xlsx 연동 시작

 'test.xlsx' 파일을 찾을 수 없습니다. (실제 평가 시 파일이 있으면 정상 작동합니다.)
